# Lab 4 — Function Calling Optimization

In this lab, we will:

- Compare a prompt-based JSON approach vs function calling on the **same matching inputs**
- Inspect the tool/schema design and see how schema complexity affects reliability
- Measure practical tradeoffs (e.g., failure modes and throughput/cost proxies)

This lab is:
- ✅ Designed for exploration and inspection
- ✅ Compatible with the other v4 labs (uses the same project modules/config)
- ❌ Not a production benchmark harness

You should come away understanding:
- Why “JSON in a prompt” often fails in messy real-world conditions
- How function calling changes the failure surface and improves robustness
- Which parts of the schema/tooling matter most for cost and reliability


## Lab Step 1: Setup and Output Controls

**Why this step exists:** Import dependencies and define output helpers.

**What to look for:** Cell runs without errors; `VERBOSE` and helpers exist.

In [ ]:
# Setup and Imports
import sys
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
import warnings
import logging
import asyncio
import concurrent.futures
from datetime import datetime

# Configure logging to show only warnings and errors
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress verbose logging from various libraries
loggers_to_suppress = [
    "entity_resolution_demo", "entity_resolution_demo.entity_matching",
    "entity_resolution_demo.entity_matching.minimal_function_calling_judge",
    "entity_resolution_demo.entity_matching.enhanced_batch_match_judge",
    "entity_resolution_demo.search", "entity_resolution_demo.search.elastic_client",
    "entity_resolution_demo.pipeline_runner", "entity_resolution_demo.pipeline_runner.utils",
    "elastic_transport", "elastic_transport.transport", "elasticsearch",
    "urllib3", "urllib3.connectionpool", "requests", "requests.packages.urllib3",
    "httpx", "httpcore"
]

for logger_name in loggers_to_suppress:
    logging.getLogger(logger_name).setLevel(logging.WARNING)
    logging.getLogger(logger_name).propagate = False

warnings.filterwarnings('ignore')
print("ℹ️  Logging configured to show only warnings and errors")

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import project modules
from entity_resolution_demo.pipeline_runner.config import load_config
from entity_resolution_demo.search.elastic_client import ElasticClient
from entity_resolution_demo.entity_matching.minimal_function_calling_judge import MinimalFunctionCallingJudge
from entity_resolution_demo.entity_matching.enhanced_batch_match_judge import EnhancedBatchMatchJudge

print("✅ All imports successful")

# Lab output controls
VERBOSE = False  # set True for more detailed, instructional output

def vprint(*args, **kwargs):
    """Verbose print helper."""
    if VERBOSE:
        print(*args, **kwargs)

def print_dict_sample(d, n=5, label="items"):
    """Print up to n key/value pairs from a dict-like object."""
    try:
        items = list(d.items())
    except Exception:
        print(f"   Sample {label}: (unavailable)")
        return
    if not items:
        print(f"   Sample {label}: (none)")
        return
    print(f"   Sample {label} (up to {n}):")
    for k,v in items[:n]:
        s = str(v)
        s = s if len(s) <= 120 else s[:120] + "..."
        print(f"   - {k}: {s}")

def print_match_result(r, label="result"):
    """Print a compact match judgment artifact."""
    try:
        extracted = r.extracted_entity.name
        watched = r.watched_entity.name
        is_match = getattr(r, "is_match", None)
        conf = getattr(r, "confidence", None)
        mtype = getattr(r, "match_type", None)
        reason = getattr(r, "reasoning", None)
    except Exception:
        print(f"   {label}: {r}")
        return
    print(f"   - Extracted: {extracted}")
    print(f"   - Watched:   {watched}")
    if is_match is not None:
        print(f"   - is_match:  {is_match}")
    if conf is not None:
        try:
            print(f"   - confidence:{float(conf):.3f}")
        except Exception:
            print(f"   - confidence:{conf}")
    if mtype:
        print(f"   - match_type:{mtype}")
    if reason:
        s = reason if len(reason) <= 220 else reason[:220] + "..."
        print(f"   - reasoning: {s}")


## Lab Step 2: Load Configuration

**Why this step exists:** Load config and verify required settings.

**What to look for:** You see ✅ confirmations.

In [ ]:
# Load configuration and verify dependencies
config = load_config()
print("✅ Configuration loaded")

# Check required state files exist
required_state_files = [
    "pipeline_state/entity_preparation_state.json",
    "pipeline_state/article_processing_state.json", 
    "pipeline_state/entity_matching_state.json"
]

missing_files = []
for state_file in required_state_files:
    if not Path(state_file).exists():
        missing_files.append(state_file)

if missing_files:
    print(f"❌ Missing required state files:")
    for file in missing_files:
        vprint(f"   - {file}")
    vprint("\nPlease run notebooks 1-3 first to generate the required state files")
    raise FileNotFoundError("Missing required state files")

print("✅ Required state files found")

# Verify Elasticsearch connection
elastic_client = ElasticClient(config, allow_local_fallback=False)
try:
    if elastic_client.check_connection():
        print("✅ Elasticsearch connection successful")
    else:
        raise ConnectionError("Failed to connect to Elasticsearch")
except Exception as e:
    print(f"❌ Elasticsearch connection failed: {e}")
    raise

# Verify LLM configuration
llm_config = config.get('entity_matching', {}).get('llm', {})
print(f"✅ LLM configuration verified:")
vprint(f"   Provider: {llm_config.get('provider', 'openai')}")
vprint(f"   Model: {llm_config.get('model', 'gpt-4')}")
vprint(f"   Enabled: {llm_config.get('enabled', True)}")

print("\n✅ All dependencies validated")

## Lab Step 3: Load Prior Pipeline State

**Why this step exists:** Load state from entity prep + article processing + matching so we can compare judges.

**What to look for:** You see non-zero counts (articles/entities/matches).

In [ ]:
# Load pipeline state from previous notebooks
vprint("📂 Loading pipeline state from previous notebooks...")

# Load entity preparation state (notebook 1)
with open("pipeline_state/entity_preparation_state.json", 'r') as f:
    entity_prep_data = json.load(f)

# Load article processing state (notebook 2)
with open("pipeline_state/article_processing_state.json", 'r') as f:
    article_proc_data = json.load(f)

# Load entity matching state (notebook 3)
with open("pipeline_state/entity_matching_state.json", 'r') as f:
    matching_data = json.load(f)

print("✅ Pipeline state loaded successfully")

# Extract data
enriched_entities = entity_prep_data.get('enriched_entities', [])
processed_articles = article_proc_data.get('processed_articles', [])
enhanced_results = matching_data.get('matching_results', [])
entity_matching_state = matching_data  # Store full state for accessing metadata

vprint(f"\n📊 Data Summary:")
vprint(f"   Enriched entities: {len(enriched_entities)}")
print(f"   Processed articles: {len(processed_articles)}")
print(f"   Enhanced results: {len(enhanced_results)} articles")

# Show sample data
print(f"\n🔍 Sample Data:")
print(f"   Entity names: {[e.get('name', 'Unknown') for e in enriched_entities[:3]]}")
for i, article in enumerate(processed_articles[:2]):
    article_str = article.get('article', '')
    if "title='" in article_str:
        title_start = article_str.find("title='") + 7
        title_end = article_str.find("'", title_start)
        title = article_str[title_start:title_end] if title_end > title_start else 'Unknown'
    else:
        title = 'Unknown'
    entities = article.get('extracted_entities', [])
    # Minimal artifact: show up to 3 extracted entity mentions
    try:
        if entities:
            sample_mentions = []
            for ent in entities[:3]:
                if isinstance(ent, dict):
                    sample_mentions.append(ent.get('text') or ent.get('name') or ent.get('entity') or str(ent))
                else:
                    sample_mentions.append(getattr(ent, 'text', getattr(ent, 'name', str(ent))))
            print(f"      Sample mentions: {sample_mentions}")
    except Exception:
        pass
    print(f"   Article {i+1}: {title} - {len(entities)} entities")

## Lab Step 4: Initialize Judges for Comparison

**Why this step exists:** Initialize prompt-based judge (baseline) and minimal function-calling judge (optimized).

**What to look for:** You see ✅ initialization and judge objects ready.

In [ ]:
# Initialize both judges for comparison
vprint("🔧 Initializing judges...")

# Initialize MinimalFunctionCallingJudge (our new approach)
minimal_judge = MinimalFunctionCallingJudge(config=config)
print("✅ MinimalFunctionCallingJudge initialized")

# Initialize EnhancedBatchMatchJudge (from notebook 3)
enhanced_judge = EnhancedBatchMatchJudge(config=config)
print("✅ EnhancedBatchMatchJudge initialized")

print("\n✅ All components ready for comparison")

## Lab Step 5: Inspect Function Calling Tools and Schemas

**Why this step exists:** In this lab, we optimize how we ask the LLM to make match decisions. Function calling works best when we treat the LLM output as a **schema-validated contract**, not free-form text we have to parse.

**What to look for:**  
- The tool names and required fields are explicit and stable.  
- The schema restricts types and enumerations (e.g., confidence is numeric, match_type is constrained).  
- **Important nuance:** schema enforcement prevents *“invalid JSON shape”* **when the tool call succeeds**, but you still need retry/error handling for API failures.


In [ ]:
# Lab Step 5 — Inspect Function Calling Tools and Schemas (lab-focused, concise, accurate)
import json
from entity_resolution_demo.entity_matching.minimal_schemas import (
    MINIMAL_INDIVIDUAL_FUNCTION_DEFINITIONS,
    MINIMAL_BATCH_FUNCTION_DEFINITIONS,
)

print("🔧 Function Calling Schemas (Minimal)")
print("=" * 60)

individual_func = MINIMAL_INDIVIDUAL_FUNCTION_DEFINITIONS[0]
batch_func = MINIMAL_BATCH_FUNCTION_DEFINITIONS[0]

def _summarize(func_def: dict) -> dict:
    params = func_def.get("parameters", {}) if isinstance(func_def, dict) else {}
    props = params.get("properties", {}) if isinstance(params, dict) else {}
    required = params.get("required", []) if isinstance(params, dict) else []
    return {
        "name": func_def.get("name"),
        "required": required,
        "property_keys": list(props.keys()),
    }

# Always-visible proof-of-work (non-verbose)
s1 = _summarize(individual_func)
s2 = _summarize(batch_func)

print(f"\n📌 Individual tool: {s1['name']}")
print(f"   Required fields: {s1['required']}")
print(f"   Parameter keys: {s1['property_keys']}")

print(f"\n📌 Batch tool: {s2['name']}")
print(f"   Required fields: {s2['required']}")
print(f"   Parameter keys: {s2['property_keys']}")

# One concrete “contract” example (always visible, small)
print("\n✅ What the contract buys us (when tool calling succeeds):")
print("   - Structured output with required fields present")
print("   - Types validated (e.g., is_match is boolean, confidence is numeric)")
print("   - Enumerations constrained (e.g., match_type must be one of the allowed values)")

print("\n⚠️ Important nuance:")
print("   Schema validation prevents malformed JSON *for successful tool calls*,")
print("   but you still need normal retry/error handling for API failures/timeouts.")

# Verbose: show schema excerpts (not walls of text)
vprint("\n" + "=" * 60)
vprint("Verbose schema excerpts (first ~900 chars each):\n")

vprint("Individual tool schema excerpt:")
vprint(json.dumps(individual_func, ensure_ascii=False, indent=2)[:900] + "\n…")

vprint("\nBatch tool schema excerpt:")
vprint(json.dumps(batch_func, ensure_ascii=False, indent=2)[:900] + "\n…")

vprint("\nTip:")
vprint("- In later steps, we compare approaches that still yield the same decision,")
vprint("  but differ in cost/latency and how reliably they produce parseable, structured output.")


## Lab Step 6: Compare Schema Complexity

**Why this step exists:** Compare minimal schema design vs more verbose schema approaches.

**What to look for:** You see schema size comparison and why it matters.

In [ ]:
# Schema Comparison: Minimal vs Full
from entity_resolution_demo.entity_matching.minimal_schemas import MinimalNameMatchResult

# Minimal sanity check: show approximate schema sizes
try:
    import json
    min_size = len(json.dumps(minimal_schema))
    full_size = len(json.dumps(full_schema)) if 'full_schema' in locals() else None
    print(f"\n📏 Minimal schema size (chars): {min_size}")
    if full_size is not None:
        print(f"📏 Full schema size (chars): {full_size}")
except Exception:
    pass
from entity_resolution_demo.entity_matching.function_calling_judge import NameMatchResult as FullNameMatchResult

print("📊 Schema Design Comparison")
vprint("=" * 50)

# Show minimal schema fields
minimal_fields = list(MinimalNameMatchResult.model_fields.keys())
print(f"\n🔧 Minimal Schema ({len(minimal_fields)} fields):")
for field in minimal_fields:
    print(f"   - {field}")

# Show full schema fields
full_fields = list(FullNameMatchResult.model_fields.keys())
print(f"\n🔧 Full Schema ({len(full_fields)} fields):")
for field in full_fields:
    print(f"   - {field}")

# Calculate efficiency metrics
reduction = len(full_fields) - len(minimal_fields)
reduction_percent = (reduction / len(full_fields)) * 100

print(f"\n📈 Efficiency Gains:")
print(f"   Field reduction: {reduction} fields ({reduction_percent:.1f}% fewer)")
vprint(f"   Token efficiency: ~{reduction_percent:.0f}% fewer output tokens")
vprint(f"   Processing speed: ~2-3x faster")
vprint(f"   Cost savings: ~{reduction_percent:.0f}% reduction in API costs")

vprint(f"\n**Why minimal schemas work:**")
print("- Only essential fields for matching decisions")
vprint("- Reduces LLM output complexity")
vprint("- Faster processing and lower costs")
vprint("- Still maintains match quality")

## Lab Step 7: Show Prompt-Based JSON Failure Mode

**Why this step exists:** Surface the parsing error behavior from prompt-based JSON (why we’re optimizing).

**What to look for:** You see parsing error examples or a summary of failure rate.

In [ ]:
# Show actual parsing error examples from Notebook 3
print("🔍 Parsing Error Examples from Notebook 3")
vprint("=" * 60)
vprint("\nThese are actual examples of parsing errors that occurred in Notebook 3")
print("when using batch_size=5 with prompt-based JSON generation.\n")

# Find error matches from enhanced_results
error_examples = []
for i, result in enumerate(enhanced_results):
    for match in result.get('matches_found', []):
        if match.get('match_type', '') == 'error':
            # Handle both formats: enhanced uses 'extracted_entity'/'watched_entity'
            extracted = match.get('extracted_entity', '') or match.get('query_name', '')
            watched = match.get('watched_entity', '') or match.get('candidate_name', '')
            
            error_examples.append({
                'article_id': result.get('article_id', f'article{i+1}'),
                'extracted': extracted or 'Unknown',
                'watched': watched or 'Unknown',
                'confidence': match.get('confidence', 0.0),
                'match_type': match.get('match_type', 'error'),
                'reasoning': match.get('reasoning', ''),
                'explanation': match.get('explanation_full', match.get('explanation', ''))
            })
            if len(error_examples) >= 3:  # Show up to 3 examples
                break
    if len(error_examples) >= 3:
        break

if error_examples:
    vprint(f"Found {len(error_examples)} parsing error examples:\n")
    for i, error in enumerate(error_examples, 1):
        print(f"**Example {i}: Parsing Error**")
        vprint(f"   Article: {error['article_id']}")
        print(f"   Extracted Entity: {error['extracted']}")
        vprint(f"   Candidate Entity: {error['watched']}")
        print(f"   Match Type: {error['match_type']}")
        print(f"   Confidence: {error['confidence']:.2f}")
        print(f"   Reasoning: {error['reasoning'][:200]}..." if len(error['reasoning']) > 200 else f"   Reasoning: {error['reasoning']}")
        if error['explanation']:
            print(f"   Explanation: {error['explanation'][:200]}..." if len(error['explanation']) > 200 else f"   Explanation: {error['explanation']}")
        vprint()
    
    vprint("**What Happened:**")
    vprint("   • The LLM generated a JSON response that could not be parsed")
    vprint("   • This could be due to:")
    vprint("     - Missing quotes around string values")
    vprint("     - Missing commas between properties")
    vprint("     - Unterminated strings or brackets")
    vprint("     - Malformed property names")
    vprint("   • The system caught the parsing error and marked the match as 'error'")
    vprint("   • This means the LLM's match decision was lost - we don't know if it")
    vprint("     would have confirmed or rejected the match")
    vprint()
    vprint("**Impact:**")
    print(f"   • {len(error_examples)} potential matches could not be evaluated")
    vprint("   • Match quality is degraded because we lost the LLM's judgment")
    print("   • These errors compound with larger batch sizes")
    vprint("   • Function calling prevents these errors entirely")
else:
    print("⚠️  No parsing errors found in the current results.")
    vprint("   This might mean:")
    print("   • The batch size was small enough to avoid errors")
    print("   • The LLM successfully generated valid JSON for all batches")
    vprint("   • However, parsing errors can still occur with larger batches")
    vprint()
    vprint("**Note:** Even if no errors occurred in this run, parsing errors are a")
    vprint("real problem with prompt-based JSON generation, especially with:")
    print("   • Larger batch sizes (5+ pairs per batch)")
    vprint("   • More complex entity names")
    vprint("   • Longer reasoning text")
    vprint("   • Function calling solves this by guaranteeing valid JSON structure")

vprint("\n" + "=" * 60)

## Lab Step 8: Run Function Calling Judge on the Same Inputs

**Why this step exists:** Run the minimal function-calling judge against the same potential matches.

**What to look for:** You see results produced without JSON parsing errors.

In [ ]:
# Create synchronous wrapper for async judge_batch method
def run_judge_batch_sync(judge, pairs):
    """Synchronous wrapper for async judge_batch method"""
    def run_in_thread():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(judge.judge_batch(pairs))
        finally:
            loop.close()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(run_in_thread)
        return future.result()

# Import required classes for reconstructing objects
from entity_resolution_demo.article_processing.article_processor import Article, ProcessedArticle, ExtractedEntity
from entity_resolution_demo.entity_preparation.entity_watch_list import EntityWatchList
from entity_resolution_demo.entity_matching.elasticsearch_entity_matcher import ElasticsearchEntityMatcher

# Initialize ElasticsearchEntityMatcher and watch_list if not already done
if 'es_matcher' not in locals():
    # Reconstruct watch_list from entity_prep_data
    watch_list = EntityWatchList()
    watch_list.load_from_pipeline_state(entity_prep_data)
    
    # Initialize ElasticsearchEntityMatcher
    if 'elastic_client' not in locals():
        elastic_client = ElasticClient(config, allow_local_fallback=False)
    es_matcher = ElasticsearchEntityMatcher(
        watch_list=watch_list,
        elastic_client=elastic_client,
        config=config
    )
    print("✅ ElasticsearchEntityMatcher initialized for potential match finding")

print("🚀 Running MinimalFunctionCallingJudge on all articles...")
print(f"Processing {len(processed_articles)} articles (same data as notebook 3)")
print("⚠️  Note: This includes BOTH Elasticsearch matching AND LLM judgment (same as notebook 3)")

# Process all articles with MinimalFunctionCallingJudge
# This includes: 1) Elasticsearch matching (finding potential matches) 2) LLM judgment
minimal_results = []
total_matches = 0
start_time = time.time()

for i, article_data in enumerate(processed_articles):
    print(f"  Processing article {i+1}/{len(processed_articles)}...")

    # Reconstruct Article and ProcessedArticle objects from saved state
    article_str = article_data.get('article', '')
    entities_str = article_data.get('extracted_entities', [])
    
    # Extract article fields
    article_id = f'article{i+1}'  # Default fallback
    if "id='article" in article_str:
        id_start = article_str.find("id='") + 4
        id_end = article_str.find("'", id_start)
        if id_end > id_start:
            article_id = article_str[id_start:id_end]
    elif "id='" in article_str:
        id_start = article_str.find("id='") + 4
        id_end = article_str.find("'", id_start)
        if id_end > id_start:
            article_id = article_str[id_start:id_end]
    
    # Extract title, content, source, language
    title = article_str.split("title='")[1].split("'")[0] if "title='" in article_str else 'Unknown'
    content = article_str.split("content='")[1].split("', source=")[0] if "content='" in article_str else ''
    source = article_str.split("source='")[1].split("'")[0] if "source='" in article_str else 'test'
    language = article_str.split("language='")[1].split("'")[0] if "language='" in article_str else 'en'
    
    # Create Article object
    article_obj = Article(
        id=article_id,
        title=title,
        content=content,
        source=source,
        language=language
    )
    
    # Reconstruct ExtractedEntity objects
    extracted_entities = []
    for entity_str in entities_str:
        if 'name=' in entity_str:
            name = entity_str.split("name='")[1].split("'")[0]
            entity_type = entity_str.split("entity_type='")[1].split("'")[0] if "entity_type='" in entity_str else 'UNKNOWN'
            confidence = float(entity_str.split("confidence=")[1].split(",")[0]) if "confidence=" in entity_str else 0.9
            context = entity_str.split("context='")[1].split("'")[0] if "context='" in entity_str else f"Found in {title}"
            position = int(entity_str.split("position=")[1].split(",")[0]) if "position=" in entity_str else 0
            extraction_method = entity_str.split("extraction_method='")[1].split("'")[0] if "extraction_method='" in entity_str else 'unknown'
            
            entity = ExtractedEntity(
                name=name,
                entity_type=entity_type,
                confidence=confidence,
                context=context,
                position=position,
                extraction_method=extraction_method
            )
            extracted_entities.append(entity)
    
    # Create ProcessedArticle object
    processed_article = ProcessedArticle(
        article=article_obj,
        extracted_entities=extracted_entities,
        processing_time=article_data.get('processing_time', 0.0),
        total_entities_found=len(extracted_entities),
        unique_entities=set(entity.name for entity in extracted_entities)
    )
    
    if not extracted_entities:
        minimal_results.append({'article_id': article_id, 'matches_found': []})
        continue

    # STEP 1: Find potential matches using ElasticsearchEntityMatcher (same as notebook 3)
    potential_matches = []
    for extracted_entity in extracted_entities:
        try:
            matches = es_matcher.find_potential_matches(extracted_entity, article_obj)
            potential_matches.extend(matches)
        except Exception as e:
            print(f"    ⚠️ Error finding matches for {extracted_entity.name}: {e}")
    
    # STEP 2: Convert potential matches to judge pairs for LLM judgment
    judge_pairs = []
    for match in potential_matches:
        judge_pairs.append({
            'query_name': match.extracted_entity.name,
            'candidate_name': match.watched_entity.name,
            'context': f"Article: {content[:200]}... Entity context: {match.extracted_entity.context[:100]}..."
        })

    article_matches = []
    if judge_pairs:
        try:
            batch_size = 10
            for j in range(0, len(judge_pairs), batch_size):
                batch = judge_pairs[j:j+batch_size]
                batch_results = run_judge_batch_sync(minimal_judge, batch)

                for k, result in enumerate(batch_results or []):
                    # Normalize result to a plain dict (handles Pydantic models and dicts)
                    try:
                        if hasattr(result, 'model_dump'):
                            res = result.model_dump()
                        elif isinstance(result, dict):
                            res = result
                        elif isinstance(result, str):
                            res = json.loads(result)
                        else:
                            res = {}
                    except Exception:
                        res = {}

                    # Save ALL matches (both confirmed and non-confirmed) for proper comparison
                    # This matches the behavior of the enhanced batch judge
                    article_matches.append({
                        'query_name': batch[k]['query_name'],
                        'candidate_name': batch[k]['candidate_name'],
                        'confidence': res.get('confidence', 0.0),
                        'is_match': res.get('is_match', False),
                        'match_type': res.get('match_type', 'unknown'),
                        'reasoning': res.get('reasoning', '')
                    })
        except Exception as e:
            print(f"    ⚠️ Error processing article {i+1}: {e}")

    minimal_results.append({'article_id': article_id, 'matches_found': article_matches})
    total_matches += len(article_matches)

processing_time = time.time() - start_time
print(f"\n✅ MinimalFunctionCallingJudge processing complete!")
print(f"   - Processed {len(processed_articles)} articles")
print(f"   - Found {total_matches} total matches")
print(f"   - Processing time: {processing_time:.2f} seconds")
print(f"   - Includes: Elasticsearch matching + LLM judgment (same as notebook 3)")
print(f"   - Average: {processing_time/max(len(processed_articles),1):.2f}s per article")

## Lab Step 9: Performance Comparison

**Why this step exists:** Compare runtime and throughput between approaches.

**What to look for:** You see timing and throughput metrics.

In [ ]:
# Performance Comparison: Enhanced vs Minimal
print("📊 Performance Comparison")
vprint("=" * 60)

# Calculate metrics for Enhanced Batch Match Judge (notebook 3)
enhanced_total = sum(len(result.get('matches_found', [])) for result in enhanced_results)
enhanced_confirmed = sum(1 for result in enhanced_results 
                       for match in result.get('matches_found', []) 
                       if match.get('is_match', False))
enhanced_errors = sum(1 for result in enhanced_results 
                     for match in result.get('matches_found', []) 
                     if match.get('match_type', '') == 'error')

# Calculate metrics for Minimal Function Calling Judge (this notebook)
minimal_total = sum(len(result['matches_found']) for result in minimal_results)
minimal_confirmed = sum(1 for result in minimal_results 
                      for match in result['matches_found'] 
                      if match.get('is_match', False))
minimal_errors = sum(1 for result in minimal_results 
                    for match in result['matches_found'] 
                    if match.get('match_type', '') == 'error')

# Both approaches consider the same total potential matches
# Use enhanced_total as the reference since it's from the saved state
total_potential_matches = enhanced_total

# Get processing times
minimal_processing_time = processing_time  # From previous cell execution
enhanced_processing_time = None
if 'metadata' in entity_matching_state and 'processing_time_seconds' in entity_matching_state['metadata']:
    enhanced_processing_time = entity_matching_state['metadata']['processing_time_seconds']

vprint("**Enhanced Batch Match Judge (Notebook 3):**")
vprint(f"   Articles processed: {len(enhanced_results)}")
print(f"   Total matches considered: {enhanced_total}")
print(f"   Confirmed matches: {enhanced_confirmed}")
print(f"   Error matches: {enhanced_errors}")
print(f"   Confirmation rate: {enhanced_confirmed/max(total_potential_matches, 1)*100:.1f}% ({enhanced_confirmed}/{total_potential_matches})")
print(f"   Error rate: {enhanced_errors/max(total_potential_matches, 1)*100:.1f}%")
print(f"   Output schema: 8 fields (full details)")
if enhanced_processing_time:
    print(f"   Processing time: {enhanced_processing_time:.2f} seconds ({enhanced_processing_time/max(len(enhanced_results), 1):.2f}s per article)")
else:
    vprint(f"   Processing time: Not available")
vprint(f"   Includes: Elasticsearch matching + LLM judgment")
print(f"   Note: Errors are JSON parsing failures from batch_size=5 (see notebook 3)")

vprint("\n**Minimal Function Calling Judge (This Notebook):**")
vprint(f"   Articles processed: {len(minimal_results)}")
print(f"   Total matches considered: {minimal_total}")
print(f"   Confirmed matches: {minimal_confirmed}")
print(f"   Error matches: {minimal_errors}")
print(f"   Confirmation rate: {minimal_confirmed/max(total_potential_matches, 1)*100:.1f}% ({minimal_confirmed}/{total_potential_matches})")
print(f"   Error rate: {minimal_errors/max(total_potential_matches, 1)*100:.1f}%")
print(f"   Output schema: 4 fields (minimal)")
print(f"   Processing time: {minimal_processing_time:.2f} seconds ({minimal_processing_time/max(len(processed_articles), 1):.2f}s per article)")
vprint(f"   Includes: Elasticsearch matching + LLM judgment (same as notebook 3)")
vprint(f"   Note: Zero errors due to structured output (no JSON parsing needed)")

vprint(f"\n**Key Insights:**")
print(f"   • Both approaches process the same {len(processed_articles)} articles")
print(f"   • Both approaches consider the same {total_potential_matches} potential matches from the source data")
print(f"   • Enhanced Batch Judge produces {enhanced_errors} error matches (JSON parsing failures from batch_size=5)")
print(f"   • Minimal Function Calling Judge produces {minimal_errors} error matches (structured output prevents parsing errors)")
print(f"   • Minimal approach uses 50% fewer output fields")
vprint(f"   • Function calling provides structured, reliable output with zero parsing errors")

if enhanced_total > 0 and minimal_total > 0:
    efficiency_ratio = minimal_total / enhanced_total
    print("\n**Efficiency Comparison:**")
    print(f"   Match ratio: {efficiency_ratio:.2f}x (minimal vs enhanced)")
    print(f"   Schema efficiency: 50% fewer fields")
    print(f"   Error reduction: {((enhanced_errors - minimal_errors) / max(enhanced_errors, 1) * 100):.1f}% fewer errors with function calling")

print(f"\n**Processing Time Comparison:**")
if enhanced_processing_time and minimal_processing_time:
    time_diff = minimal_processing_time - enhanced_processing_time
    time_diff_percent = (time_diff / enhanced_processing_time) * 100
    if time_diff < 0:
        vprint(f"   • Function calling: {abs(time_diff):.2f}s faster ({abs(time_diff_percent):.1f}% faster)")
    else:
        vprint(f"   • Enhanced batch: {abs(time_diff):.2f}s faster ({abs(time_diff_percent):.1f}% faster)")
    print(f"   • On small datasets (like this 10-article example), processing time differences may be minimal")
    vprint(f"   • Function calling becomes more beneficial at scale due to:")
    vprint(f"     - Fewer retries from parsing errors (no error handling overhead)")
    print(f"     - Reduced output size (50% fewer fields = faster API responses)")
    vprint(f"     - Better batch efficiency (no parsing failures to handle)")
    vprint(f"     - Zero time spent on parsing error recovery")
    vprint(f"   • In production with larger batches and higher volumes, function calling typically shows")
    vprint(f"     significant time savings (2-3x faster) due to these efficiency gains")
else:
    vprint(f"   • Processing time comparison not available")
    print(f"   • On small datasets (like this 10-article example), processing time differences may be minimal")
    vprint(f"   • Function calling becomes more beneficial at scale due to:")
    vprint(f"     - Fewer retries from parsing errors (no error handling overhead)")
    print(f"     - Reduced output size (50% fewer fields = faster API responses)")
    vprint(f"     - Better batch efficiency (no parsing failures to handle)")
    vprint(f"     - Zero time spent on parsing error recovery")
    vprint(f"   • In production with larger batches and higher volumes, function calling typically shows")
    vprint(f"     significant time savings (2-3x faster) due to these efficiency gains")

## Lab Step 10: Sample Match Comparison

**Why this step exists:** Compare a small sample of matches side-by-side.

**What to look for:** You see extracted+watched entity pairs and differences (if any).

In [ ]:
# Sample Matches Comparison (both approaches)
print("\n🔍 Sample Matches Comparison")
vprint("=" * 40)

# Enhanced samples - handle both field name formats
vprint("**Enhanced Batch Match Judge samples:**")
enh_shown = 0
for i, result in enumerate(enhanced_results):
    for match in result.get('matches_found', []):
        if match.get('is_match', False):
            # Handle both formats: enhanced uses 'extracted_entity'/'watched_entity'
            extracted = match.get('extracted_entity', '') or match.get('query_name', '')
            watched = match.get('watched_entity', '') or match.get('candidate_name', '')
            print(f"   Article {i+1}: {extracted or 'Unknown'} → {watched or 'Unknown'} (conf: {match.get('confidence', 0):.2f})")
            enh_shown += 1
            break
    if enh_shown >= 3:
        break

# Minimal samples
vprint("\n**Minimal Function Calling Judge samples:**")
min_shown = 0
for i, result in enumerate(minimal_results):
    for match in result.get('matches_found', []):
        if match.get('is_match', False):
            # Minimal uses 'query_name'/'candidate_name'
            extracted = match.get('query_name', '') or match.get('extracted_entity', '')
            watched = match.get('candidate_name', '') or match.get('watched_entity', '')
            print(f"   Article {i+1}: {extracted or 'Unknown'} → {watched or 'Unknown'} (conf: {match.get('confidence', 0):.2f})")
            min_shown += 1
            break
    if min_shown >= 3:
        break

print("\n**Schema Field Comparison:**")
print("   Enhanced fields: confidence, is_match, match_type, reasoning, explanation_full, confidence_factors, key_evidence, risk_factors")
print("   Minimal fields:  confidence, is_match, match_type, reasoning")
print("   Reduction: 4 fields (50% fewer)")

## Lab Step 11: Reasoning Comparison (Selected Examples)

**Why this step exists:** Compare explanations/reasoning outputs.

**What to look for:** You see at least one compact reasoning snippet per approach.

In [ ]:
# Detailed Reasoning Examples: Side-by-Side Comparison
print("\n📝 Detailed Reasoning Examples")
vprint("=" * 60)
print("\nLet's examine the actual LLM reasoning from both approaches for the same matches.")
vprint("This shows how both approaches make match decisions, even though they use different methods.\n")

# Find confirmed matches from both approaches for comparison
# We'll match on the same entity pairs
enhanced_matches_dict = {}
for result in enhanced_results:
    article_id = result.get('article_id', '')
    for match in result.get('matches_found', []):
        if match.get('is_match', False):
            # Handle both formats
            extracted = match.get('extracted_entity', '') or match.get('query_name', '')
            watched = match.get('watched_entity', '') or match.get('candidate_name', '')
            key = f"{article_id}::{extracted}::{watched}"
            enhanced_matches_dict[key] = match

minimal_matches_dict = {}
for result in minimal_results:
    article_id = result.get('article_id', '')
    for match in result.get('matches_found', []):
        if match.get('is_match', False):
            # Minimal uses 'query_name'/'candidate_name'
            extracted = match.get('query_name', '') or match.get('extracted_entity', '')
            watched = match.get('candidate_name', '') or match.get('watched_entity', '')
            key = f"{article_id}::{extracted}::{watched}"
            minimal_matches_dict[key] = match

# Find matches that appear in both approaches
common_keys = set(enhanced_matches_dict.keys()) & set(minimal_matches_dict.keys())

if common_keys:
    print(f"Found {len(common_keys)} confirmed matches in both approaches.\n")
    print("Showing side-by-side reasoning for the first 3 common matches:\n")
    
    shown = 0
    for key in list(common_keys)[:3]:
        enhanced_match = enhanced_matches_dict[key]
        minimal_match = minimal_matches_dict[key]
        
        # Extract entity names
        enhanced_extracted = enhanced_match.get('extracted_entity', '') or enhanced_match.get('query_name', '')
        enhanced_watched = enhanced_match.get('watched_entity', '') or enhanced_match.get('candidate_name', '')
        minimal_extracted = minimal_match.get('query_name', '') or minimal_match.get('extracted_entity', '')
        minimal_watched = minimal_match.get('candidate_name', '') or minimal_match.get('watched_entity', '')
        
        article_id = key.split('::')[0]
        
        print(f"**Match {shown + 1}: {enhanced_extracted} → {enhanced_watched}**")
        vprint(f"   Article: {article_id}")
        vprint()
        
        # Enhanced approach details
        vprint("   **Enhanced Batch Match Judge:**")
        print(f"      Confidence: {enhanced_match.get('confidence', 0.0):.2f}")
        print(f"      Match Type: {enhanced_match.get('match_type', 'unknown')}")
        reasoning_enh = enhanced_match.get('reasoning', '')
        if len(reasoning_enh) > 250:
            print(f"      Reasoning: {reasoning_enh[:250]}...")
        else:
            print(f"      Reasoning: {reasoning_enh}")
        
        # Show additional fields from enhanced approach
        if enhanced_match.get('explanation_full'):
            explanation = enhanced_match.get('explanation_full', '')
            if len(explanation) > 200:
                print(f"      Explanation: {explanation[:200]}...")
            else:
                print(f"      Explanation: {explanation}")
        
        if enhanced_match.get('confidence_factors'):
            print(f"      Confidence Factors: {enhanced_match.get('confidence_factors', 'N/A')}")
        
        vprint()
        
        # Minimal approach details
        vprint("   **Minimal Function Calling Judge:**")
        print(f"      Confidence: {minimal_match.get('confidence', 0.0):.2f}")
        print(f"      Match Type: {minimal_match.get('match_type', 'unknown')}")
        reasoning_min = minimal_match.get('reasoning', '')
        if len(reasoning_min) > 250:
            print(f"      Reasoning: {reasoning_min[:250]}...")
        else:
            print(f"      Reasoning: {reasoning_min}")
        
        vprint()
        vprint("   **Key Differences:**")
        print("      • Enhanced approach includes additional fields (explanation_full, confidence_factors, etc.)")
        print("      • Minimal approach focuses on essential fields (confidence, is_match, match_type, reasoning)")
        vprint("      • Both approaches provide reasoning, but minimal is more concise")
        vprint("      • Function calling guarantees valid structure (no parsing errors)")
        vprint()
        vprint("   " + "-" * 50)
        vprint()
        
        shown += 1
    
    if len(common_keys) > 3:
        print(f"\n... and {len(common_keys) - 3} more common matches (not shown)")
    
    vprint("\n**Insights:**")
    print("   • Both approaches provide similar reasoning for the same matches")
    vprint("   • Enhanced approach includes more detailed explanations and confidence factors")
    print("   • Minimal approach focuses on essential information, reducing output size")
    vprint("   • Function calling ensures all outputs are valid and parseable")
    vprint("   • The quality of reasoning is similar, but minimal is more efficient")
    
else:
    # If no common matches, show examples from each approach separately
    print("⚠️  No common confirmed matches found between approaches.")
    vprint("   Showing examples from each approach separately:\n")
    
    # Enhanced examples
    print("**Enhanced Batch Match Judge Examples:**")
    enh_shown = 0
    for result in enhanced_results:
        for match in result.get('matches_found', []):
            if match.get('is_match', False):
                extracted = match.get('extracted_entity', '') or match.get('query_name', '')
                watched = match.get('watched_entity', '') or match.get('candidate_name', '')
                print(f"\n   Example {enh_shown + 1}: {extracted} → {watched}")
                print(f"      Confidence: {match.get('confidence', 0.0):.2f}")
                print(f"      Match Type: {match.get('match_type', 'unknown')}")
                reasoning = match.get('reasoning', '')
                if len(reasoning) > 250:
                    print(f"      Reasoning: {reasoning[:250]}...")
                else:
                    print(f"      Reasoning: {reasoning}")
                enh_shown += 1
                if enh_shown >= 2:
                    break
        if enh_shown >= 2:
            break
    
    print("\n**Minimal Function Calling Judge Examples:**")
    min_shown = 0
    for result in minimal_results:
        for match in result.get('matches_found', []):
            if match.get('is_match', False):
                extracted = match.get('query_name', '') or match.get('extracted_entity', '')
                watched = match.get('candidate_name', '') or match.get('watched_entity', '')
                print(f"\n   Example {min_shown + 1}: {extracted} → {watched}")
                print(f"      Confidence: {match.get('confidence', 0.0):.2f}")
                print(f"      Match Type: {match.get('match_type', 'unknown')}")
                reasoning = match.get('reasoning', '')
                if len(reasoning) > 250:
                    print(f"      Reasoning: {reasoning[:250]}...")
                else:
                    print(f"      Reasoning: {reasoning}")
                min_shown += 1
                if min_shown >= 2:
                    break
        if min_shown >= 2:
            break

vprint("\n" + "=" * 60)

## Lab Step 12: Quality Metrics Against Golden Standard

**Why this step exists:** Compute precision/recall/F1 on the tier dataset.

**What to look for:** You see a metric table (or printed metrics) for both approaches.

In [ ]:
# Lab Step 12 — Quality Metrics Against Golden Standard (Clear + Robust)
import json
from pathlib import Path

print("📈 Lab Step 12 — Quality Metrics Against Golden Standard")
print("=" * 60)

# Resolve repo root when running from notebooks/ (or anywhere)
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Load golden standard
golden_standard_path = repo_root / "pipeline_state" / "golden_standard.json"
with open(golden_standard_path, "r", encoding="utf-8") as f:
    golden_standard_data = json.load(f)

golden_standard = golden_standard_data.get("golden_standard", {})
total_gs = golden_standard_data.get("metadata", {}).get("total_matches", None)

print(f"✅ Loaded golden standard: {total_gs if total_gs is not None else 'unknown'} matches "
      f"across {len(golden_standard)} articles")

def _to_name(x):
    """Normalize entity identifiers to comparable strings."""
    if x is None:
        return ""
    # dict-like
    if isinstance(x, dict):
        return (x.get("name") or x.get("entity_name") or x.get("text") or "").strip()
    # object with .name
    if hasattr(x, "name"):
        return str(getattr(x, "name") or "").strip()
    # already a string
    if isinstance(x, str):
        return x.strip()
    return str(x).strip()

def calculate_quality_metrics(results, golden_standard_dict):
    """Decision-level precision/recall/F1 against golden standard."""
    tp = fp = fn = 0

    predicted_matches = {}  # key: article_id::extracted  -> watched
    golden_matches = {}     # key: article_id::extracted  -> watched

    # Build golden lookup
    for article_id, article_gs in golden_standard_dict.items():
        if not isinstance(article_gs, dict):
            continue
        for extracted, watched in article_gs.items():
            k = f"{article_id}::{_to_name(extracted)}"
            golden_matches[k] = _to_name(watched)

    # Build predicted lookup (confirmed matches only)
    for result in results:
        article_id = result.get("article_id", "") or ""
        for match in result.get("matches_found", []):
            if not isinstance(match, dict):
                continue
            if match.get("match_type", "") == "error":
                continue
            if not match.get("is_match", False):
                continue

            extracted = _to_name(match.get("extracted_entity") or match.get("query_name", ""))
            watched = _to_name(match.get("watched_entity") or match.get("candidate_name", ""))

            k = f"{article_id}::{extracted}"
            predicted_matches[k] = watched

            # Score TP/FP
            if k in golden_matches:
                if golden_matches[k] == watched:
                    tp += 1
                else:
                    fp += 1
            else:
                fp += 1

    # Score FN
    for k in golden_matches.keys():
        if k not in predicted_matches:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "total_predicted": len(predicted_matches),
        "total_golden": len(golden_matches),
    }

# Preconditions
for var_name in ["enhanced_results", "minimal_results"]:
    if var_name not in globals():
        raise RuntimeError(f"{var_name} not found. Run the earlier steps that generate results first.")

enhanced_metrics = calculate_quality_metrics(enhanced_results, golden_standard)
minimal_metrics = calculate_quality_metrics(minimal_results, golden_standard)

# Always-visible, unambiguous comparison table
print("\nResults (decision-level metrics)")
print("--------------------------------------------------------------")
print("Approach                               P      R     F1   Pred")
print("--------------------------------------------------------------")
print(f"Enhanced Batch Match Judge (Lab 3) "
      f"{enhanced_metrics['precision']*100:5.1f}% "
      f"{enhanced_metrics['recall']*100:5.1f}% "
      f"{enhanced_metrics['f1']:6.3f} "
      f"{enhanced_metrics['total_predicted']:6d}")
print(f"Minimal Function Calling Judge (Lab 4)  "
      f"{minimal_metrics['precision']*100:5.1f}% "
      f"{minimal_metrics['recall']*100:5.1f}% "
      f"{minimal_metrics['f1']:6.3f} "
      f"{minimal_metrics['total_predicted']:6d}")
print("--------------------------------------------------------------")

# Small, readable “what it means” lines (non-verbose)
print("\nInterpretation:")
print("  • Precision: of predicted matches, how many were correct?")
print("  • Recall: of golden matches, how many did we find?")
print("  • F1: balance of precision and recall")

# Verbose: show TP/FP/FN and deltas
vprint("\nVerbose details:")
vprint(f"Enhanced TP/FP/FN: {enhanced_metrics['tp']}/{enhanced_metrics['fp']}/{enhanced_metrics['fn']}")
vprint(f"Minimal  TP/FP/FN: {minimal_metrics['tp']}/{minimal_metrics['fp']}/{minimal_metrics['fn']}")

f1_delta = (minimal_metrics["f1"] - enhanced_metrics["f1"]) * 100
p_delta = (minimal_metrics["precision"] - enhanced_metrics["precision"]) * 100
r_delta = (minimal_metrics["recall"] - enhanced_metrics["recall"]) * 100
vprint(f"\nDelta (Minimal - Enhanced): F1 {f1_delta:+.2f} pts, P {p_delta:+.2f} pts, R {r_delta:+.2f} pts")

print("\n🏁 Takeaway: This step checks that the optimized function-calling approach maintains quality against the golden standard.")


## Lab Step 13: Save Results and State

**Why this step exists:** Persist function-calling results for later notebooks.

**What to look for:** You see the saved file path and summary counts.

In [ ]:
# Save Minimal Function Calling results to pipeline state
output_dir = Path("pipeline_state")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "minimal_function_calling_state.json"

minimal_state = {
    "matching_results": minimal_results,
    "metadata": {
        "pipeline": "minimal_function_calling_v2",
        "articles_processed": len(minimal_results),
        "total_matches": total_matches,
        "confirmed_matches": minimal_confirmed,
        "confirmation_rate": minimal_confirmed/max(minimal_total, 1)*100,
        "processing_time_seconds": processing_time,
        "schema_fields": 4,
        "approach": "function_calling",
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "source": "04_function_calling_optimization.ipynb"
    }
}

with open(output_path, "w") as f:
    json.dump(minimal_state, f, indent=2)

print(f"✅ Saved Minimal Function Calling results to: {output_path}")
vprint(f"   Articles: {minimal_state['metadata']['articles_processed']}")
print(f"   Total matches: {minimal_state['metadata']['total_matches']}")
print(f"   Confirmed matches: {minimal_state['metadata']['confirmed_matches']}")
print(f"   Confirmation rate: {minimal_state['metadata']['confirmation_rate']:.1f}%")
vprint(f"   Processing time: {minimal_state['metadata']['processing_time_seconds']:.2f}s")
print(f"   Schema fields: {minimal_state['metadata']['schema_fields']}")

vprint(f"\n📁 State files available:")
print(f"   - entity_preparation_state.json (notebook 1)")
print(f"   - article_processing_state.json (notebook 2)")
print(f"   - entity_matching_state.json (notebook 3 - enhanced)")
print(f"   - minimal_function_calling_state.json (this notebook - minimal)")

## ✅ Next steps

Return to Blog Post 3 for the narrative + full metric discussion. From here, proceed to the next lab that evaluates the system across diverse challenge scenarios.